[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# A Small Catalog &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup with the catalog loaded and `listing` and `search` defined,
so that the tasks have an application to work on. Run it first, then the tasks in any order.


In [1]:
import json
import os
import re
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, DatabaseProxy, ForeignKeyField, IntegerField, Model,
                    OperationalError, PostgresqlDatabase, SqliteDatabase, TextField, chunked)
from playhouse.db_url import connect as connect_url
from playhouse.pool import MaxConnectionsExceeded, PooledSqliteDatabase
from playhouse.shortcuts import model_to_dict
from playhouse.sqlite_ext import FTS5Model, RowIDField, SearchField
from playhouse.test_utils import assert_query_count

CATALOG = [                                                         # author, title, year, pages, blurb
    ("Ursula Vance", "The Salt Road", 2014, 312, "A road made of salt, and the sea beside it."),
    ("Ursula Vance", "Nightjar", 2018, 244, "Birds at dusk, and a house nobody lives in."),
    ("Ursula Vance", "The Quiet Engine", 2021, 398, "Engines, quiet ones, and the sea again."),
    ("Marco Pietra", "Stone and Tide", 2009, 501, "Granite, tides, and a quarry above the sea."),
    ("Marco Pietra", "The Lantern Keeper", 2016, 276, "A lantern, a keeper, and long winters."),
    ("Marco Pietra", "Riverwork", 2022, 189, "Rivers, locks and the work of moving water."),
    ("Ines O'Brien", "A Careful Fire", 1998, 420, "Fire, carefully kept, in a cold house."),
    ("Ines O'Brien", "The Long Field", 2004, 355, "One field, many seasons, and the sea far off."),
    ("Ines O'Brien", "Winter Harbour", 2011, 263, "A harbour in winter, and the boats laid up."),
    ("Kofi Mensah", "The Drum Line", 2015, 198, "Drums, a line of them, and a city waking up."),
    ("Kofi Mensah", "Harmattan", 2019, 331, "A dry wind, and what it carries with it."),
    ("Kofi Mensah", "Small Machines", 2023, 287, "Machines, small ones, and the people who mend them."),
]

WORK = Path(tempfile.mkdtemp(prefix="catalog-"))

database = DatabaseProxy()                                          # which database is a setting


class CatalogModel(Model):
    class Meta:
        database = database


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()
    blurb = TextField()


class BookIndex(FTS5Model):
    rowid = RowIDField()
    title = SearchField()
    blurb = SearchField()

    class Meta:
        database = database
        options = {"content": Book}


def configure(url=None):
    """Point the catalog at a database. The URL is the only line that names a backend."""
    chosen = connect_url(url or os.environ.get("DATABASE_URL", f"sqlite:///{WORK / 'catalog.db'}"))
    database.initialize(chosen)
    return chosen


live = configure()
print("peewee", peewee.__version__, "| backend:", type(live).__name__)
print("models bound:", [model.__name__ for model in (Author, Book, BookIndex)])


live.create_tables([Author, Book, BookIndex])
with live.atomic():
    Author.insert_many([{"name": name} for name in sorted({a for a, *_ in CATALOG})]).execute()
    keys = {author.name: author.id for author in Author.select()}
    Book.insert_many([{"title": title, "author": keys[author], "year": year,
                       "pages": pages, "blurb": blurb}
                      for author, title, year, pages, blurb in CATALOG]).execute()
BookIndex.rebuild()


def listing(page=1, per_page=5, since=2000):
    """One page of the catalog, oldest first, with each book's author."""
    query = (Book.select(Book, Author)
                 .join(Author)
                 .where(Book.year >= since)
                 .order_by(Book.year, Book.id)
                 .paginate(page, per_page))
    return [(row.year, row.title, row.author.name) for row in query]


def search(typed, limit=5):
    """Search the catalog for whatever was typed, FTS5 behind one function."""
    query = BookIndex.web_query(typed or "")
    if not query.strip():
        return []
    return list(Book.select(Book, Author)
                    .join(Author)
                    .switch(Book)
                    .join(BookIndex, on=(Book.id == BookIndex.rowid))
                    .where(BookIndex.match(query))
                    .order_by(BookIndex.bm25())
                    .limit(limit))


print("ready:", Author.select().count(), "authors and", Book.select().count(), "books")


peewee 4.5.1 | backend: SqliteDatabase
models bound: ['Author', 'Book', 'BookIndex']
ready: 4 authors and 12 books


**1.** A route for the authors.


In [2]:
def authors_route():
    """Each author and how many books they have, in one query."""
    query = (Author.select(Author.name, peewee.fn.COUNT(Book.id).alias("books"))
                   .join(Book)
                   .group_by(Author.name)
                   .order_by(Author.name))
    return [{"name": row.name, "books": row.books} for row in query]


with assert_query_count(1):
    rows = authors_route()

print(json.dumps(rows))


[{"name": "Ines O'Brien", "books": 3}, {"name": "Kofi Mensah", "books": 3}, {"name": "Marco Pietra", "books": 3}, {"name": "Ursula Vance", "books": 3}]


`group_by` with an aliased `COUNT` is one row per author and one query for the lot, which the
assertion is there to keep true. Counting in Python would have needed every book fetched.


**2.** Two pages, with nothing on both.


In [3]:
first = listing(page=1, per_page=4)
second = listing(page=2, per_page=4)

print("page 1:", [title for _, title, _ in first])
print("page 2:", [title for _, title, _ in second])
print("on both:", sorted({title for _, title, _ in first} & {title for _, title, _ in second}))


page 1: ['The Long Field', 'Stone and Tide', 'Winter Harbour', 'The Salt Road']
page 2: ['The Drum Line', 'The Lantern Keeper', 'Nightjar', 'Harmattan']
on both: []


The `order_by` is `(year, id)`, and `id` is unique, so there are no ties for the pages to slice
through differently. Ordering by `year` alone would leave the books of one year in whatever order
came back, and a row could appear twice.


**3.** One response, three ways.


In [4]:
query = Book.select(Book.title, Book.year).where(Book.year >= 2021).order_by(Book.title)

print("dicts: ", json.dumps(list(query.dicts())))
print("tuples:", list(query.tuples()))
print("models:", [model_to_dict(book, only=[Book.title, Book.year], recurse=False)
                  for book in Book.select().where(Book.year >= 2021).order_by(Book.title)])


dicts:  [{"title": "Riverwork", "year": 2022}, {"title": "Small Machines", "year": 2023}, {"title": "The Quiet Engine", "year": 2021}]
tuples: [('Riverwork', 2022), ('Small Machines', 2023), ('The Quiet Engine', 2021)]
models: [{'title': 'Riverwork', 'year': 2022}, {'title': 'Small Machines', 'year': 2023}, {'title': 'The Quiet Engine', 'year': 2021}]


`dicts()` for an endpoint: it is already what `json.dumps` wants, and it fetched only the two
columns asked for. `tuples()` where position is the point, such as a CSV. `model_to_dict` when you
have whole models in hand already and do not want to run the query again.


**4.** A search that cannot raise.


In [5]:
for typed in ("c++", "sea AND", "-", "engines"):
    found = search(typed)
    print(f"  {typed!r:<12} {[book.title for book in found]}")


  'c++'        []
  'sea AND'    ['The Quiet Engine', 'Stone and Tide', 'The Long Field', 'The Salt Road']
  '-'          []
  'engines'    ['The Quiet Engine']


`web_query` quoted the `+`, dropped the dangling `AND` and made nothing at all of the lone hyphen.
Every one of those four returns a list, which is what a route needs: an empty page is a page, and a
traceback is not.


**5.** A database of its own.


In [6]:
before = Book.select().count()

aside = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})
with aside.bind_ctx([Author, Book]):
    aside.create_tables([Author, Book])
    writer = Author.create(name="Someone Else")
    Book.create(title="One", author=writer, year=2024, pages=100, blurb="b")
    Book.create(title="Two", author=writer, year=2025, pages=110, blurb="b")
    print("inside the block:", Book.select().count(), "books")

print("the catalog, untouched:", Book.select().count(), "books | before:", before)


inside the block: 2 books
the catalog, untouched: 12 books | before: 12


Two books inside the block and twelve outside it. `bind_ctx` moved the models for the length of the
`with` and put them back, which is exactly what a test fixture needs and why the catalog's own
database never had to be involved.


**6.** The other backend, offline.


In [7]:
elsewhere = PostgresqlDatabase(None)

with elsewhere.bind_ctx([Author, Book]):
    print("create:")
    for column in Book._schema._create_table().query()[0].split("(", 1)[1].rstrip(")").split(", "):
        print("   ", column)
    print("insert:")
    print("   ", " ".join(Book.insert(title="x", author=1, year=2024,
                                      pages=1, blurb="b").sql()[0].split()))


create:
    "id" SERIAL NOT NULL PRIMARY KEY
    "title" VARCHAR(80) NOT NULL
    "author_id" INTEGER NOT NULL
    "year" INTEGER NOT NULL
    "pages" INTEGER NOT NULL
    "blurb" TEXT NOT NULL
    FOREIGN KEY ("author_id") REFERENCES "author" ("id"
insert:
    INSERT INTO "book" ("title", "author_id", "year", "pages", "blurb") VALUES (%s, %s, %s, %s, %s) RETURNING "book"."id"


`SERIAL` where SQLite had `INTEGER`, `%s` where it had `?`, and a `RETURNING` clause that SQLite's
insert does not carry. Nothing connected to anything to print that.


---

&#8592; **Back to:** [A Small Catalog](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/12-a-small-catalog.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
